In [1]:
!nvidia-smi

Sun Jun 21 14:09:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-PCIE-40GB          On  |   00000000:86:00.0 Off |                    0 |
| N/A   33C    P0             33W /  250W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import xarray as xr

# path = "/home/orabe/fNIRS_sparseToDense/datasets_14062026/processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_4_test.nc"
# path = "datasets_14062026/processed/Anderson_sparse/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs_rest_4_test.nc"
path = "datasets/processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_0.nc"

data = xr.open_dataarray(path)
data.sizes


Frozen({'parcel': 143, 'chromo': 2, 'time': 174})

In [3]:
import os
import numpy as np
import xarray as xr
import pickle
import glob
import warnings
from pathlib import Path, PureWindowsPath
import matplotlib.pyplot as plt
import re

import cedalion
import cedalion.sigproc.motion as motion_correct
import cedalion.sigproc.quality as quality
import cedalion.sigproc.physio as physio
import cedalion.dot as dot
# NEW: helper-only alpha_meas estimation API for recon augmentation
import cedalion.dot.image_recon as dot_image_recon
import cedalion.nirs as nirs
import cedalion.vis.anatomy
from cedalion.io.forward_model import load_Adot

from cedalion import units
import pandas as pd

warnings.filterwarnings("ignore")

In [4]:
def get_bad_ch_mask(int_data, ch_preproc) -> list:
    # Saturated and Dark Channels

    dark_sat_thresh = [1e-3, 0.84]
    amp_threshs_sat = [0., dark_sat_thresh[1]]
    amp_threshs_low = [dark_sat_thresh[0], 1]
    _, amp_mask_sat = quality.mean_amp(int_data, amp_threshs_sat)
    _, amp_mask_low = quality.mean_amp(int_data, amp_threshs_low)
    _, snr_mask = quality.snr(int_data, 10)
    amp_mask=amp_mask_sat & amp_mask_low

    _, list_bad_ch = quality.prune_ch(int_data, [amp_mask, snr_mask], "all")
   
    return list_bad_ch

# Theekshana
# def get_bad_ch_mask(int_data: xr.DataArray, ch_preproc: dict) -> list:
#     # SCI and PSP Mask    
#     sci, sci_mask = quality.sci(int_data, ch_preproc['window_len'], ch_preproc['sci_thresh'])
#     psp, psp_mask = quality.psp(int_data, ch_preproc['window_len'], ch_preproc['psp_thresh'])

#     sci_psp_mask=sci_mask & psp_mask
#     perc_time_clean = sci_psp_mask.sum(dim="time") / len(sci.time)

#     scipsp_bad_ch=[]
#     for ch in perc_time_clean.channel.values:
#         if perc_time_clean.sel(channel=ch).values < ch_preproc['perc_time_clean']: # dont make the mistake of using the inv condition >:| 
#             scipsp_bad_ch.append(ch)

#     sum_bad_ch = scipsp_bad_ch
#     list_bad_ch = sorted(list(set(sum_bad_ch)))  # remove duplicates

#     print("Flagged Channels : ",len(list_bad_ch), '/', len(int_data.channel))
#     print("Percentage: ", int(len(list_bad_ch) / len(int_data.channel) * 100), '%')
    
#     return list_bad_ch

In [5]:
def standardize_trial_types(DATASET_NAME: str, file: str, stim: pd.DataFrame, rec):
    
    if DATASET_NAME == "FreshMotor":
        # map trial types to left or right depending on the name of the file
        m = re.search(r'(?i)(left|right)', file)

        # rename from MOTOR to left/right
        rec.stim.trial_type = m.group(1).lower()
        # rec.stim = stim
    
    elif DATASET_NAME == "BallSqueezingHD":
        mapping = {
            "Right": "right", # BallSqueezingHD
            "Left": "left",   # BallSqueezingHD
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME == "BS_Laura":
        stim = stim.copy()
        # stim["duration"] = 10.0  # NEW FIX: BS_Laura events are treated as 10s, not the raw 5s annotation
        rec.stim = stim
        
    elif DATASET_NAME == "Electrical_Thermal":
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
    
    rec.stim.sort_values(by="onset", ignore_index=True, inplace=True)

    # attach/update stim info to rec
    # rec.stim = stim

    return stim, rec

# def standardize_trial_types(DATASET_NAME: str, file: str, rec):
    
#     if DATASET_NAME == "FreshMotor":
#         # map trial types to left or right depending on the name of the file
#         m = re.search(r'(?i)(left|right)', file)

#         # rename from MOTOR to left/right
#         rec.stim.trial_type = m.group(1).lower()
    
#     else:
#         mapping = {
#             "Right": "right", # BallSqueezingHD
#             "Left": "left",   # BallSqueezingHD
#             "ElectricalVAS7": "right", # TODO: Electrical_Thermal
#             "ElectricalVAS3": "left",  # TODO: Electrical_Thermal
#         }
#         rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)

#     return rec


In [6]:
# "full", "subset_2" for spatial_sampling, "motor" for motor_sampling

# subset_type = ""


In [7]:
augmentation_strategy = "imageRecon_params"

mixed_sparse = False
if mixed_sparse:
    n_chs = 50
    subset_type = f"motor_{n_chs}chs" # for sparsified version of Laura
    sparsified_data_path = f'datasets/pre_processed/BS_Laura/channel_subset_BS_Laura_k=2_c3c4k=51_dist4.5_{n_chs}chs.npy'
else:
    subset_type = "full"

base_path = "/home/orabe/fNIRS_sparseToDense/"

# Available datasets:
# DATASET_NAME = "BallSqueezingHD_modified"
# DATASET_NAME = "FreshMotor"
# DATASET_NAME = "BS_Laura"
# DATASET_NAME = "ElectricalThermal"
DATASET_NAME = "vfc_hd"
# DATASET_NAME = "Anderson_sparse"

raw_path = Path(f'datasets/raw/{DATASET_NAME}')

pre_processed_path = Path(f'datasets/pre_processed/{augmentation_strategy}/{DATASET_NAME}/{subset_type}')

pre_processed_path.mkdir(parents=True, exist_ok=True)
pre_processed_path


PosixPath('datasets/pre_processed/imageRecon_params/vfc_hd/full')

In [8]:
# --- NEW: recon parameter augmentation config ---
k_alpha_meas = 0.01
alpha_meas_multipliers = [0.1, 1.0, 10.0]
alpha_spatial_multipliers = [0.1, 1.0, 10.0]
baseline_alpha_spatial = 1e-2


def get_recon_settings(c_meas):
    try:
        alpha_meas_0 = dot_image_recon.estimate_alpha_meas(c_meas, K=k_alpha_meas)
    except Exception:
        # fallback for older Cedalion versions: alpha_meas = K / median(C_meas)
        alpha_meas_0 = k_alpha_meas / np.median(c_meas.values)

    alpha_spatial_0 = float(baseline_alpha_spatial)
    settings = []
    for m_meas in alpha_meas_multipliers:
        for m_spatial in alpha_spatial_multipliers:
            settings.append(
                (
                    f"am_{m_meas:g}__as_{m_spatial:g}",
                    float(alpha_meas_0 * m_meas),
                    float(alpha_spatial_0 * m_spatial),
                    alpha_meas_0,
                    alpha_spatial_0,
                    float(m_meas),
                    float(m_spatial),
                )
            )

    return settings


def get_param_config_folder(alpha_meas_multiplier, alpha_spatial_multiplier):
    return f"am_{float(alpha_meas_multiplier):g}__as_{float(alpha_spatial_multiplier):g}"

pre_processed_path


PosixPath('datasets/pre_processed/imageRecon_params/vfc_hd/full')

In [9]:
if DATASET_NAME == "BallSqueezingHD_modified":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"

elif DATASET_NAME == "BS_Laura":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"
    
elif DATASET_NAME == "Electrical_Thermal":
    raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-Electrical*_nirs.snirf"
    # TODO: exclude subjects without txt files for landmarks coords
    
elif DATASET_NAME == "FreshMotor":
    duration = "*" # * to include both 2s and 3s
    raw_dir = f"{raw_path}/sub-*/ses-*{duration}/nirs/sub-*_ses-*{duration}_task-FRESHMOTOR_nirs.snirf"
elif DATASET_NAME == "vfc_hd":
    # "datasets/raw/vfc_hd/sub-01/nirs/sub-01_ses-02_task-WordStroop_run-01_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
elif DATASET_NAME == "Anderson_sparse":
    # raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"    

else:
    raise ValueError("Unknown dataset name")

files = glob.glob(raw_dir)

# TODO: to be confirmed
# remove non-BS files for Laura's dataset to avoid errors
if DATASET_NAME == "BS_Laura":
    files = [p for p in files if "BS" in os.path.basename(p)]
    # remove files that has this pattern in the name: _acq-4NN_nirs:
    files = [p for p in files if "_acq-4NN_nirs" not in os.path.basename(p)]
    
files = sorted(files)
print(f"{len(files)} files found.")

17 files found.


In [10]:
files[0]

'datasets/raw/vfc_hd/sub-01/nirs/sub-01_ses-02_task-WordStroop_run-01_nirs.snirf'

In [11]:
# spatial_sampling is the approach we use to subsample channels based on predifined spatial locations. See src/subset/optode_subsets.ipynb
spatial_sampling = False
if spatial_sampling:
    # This is a temoraly code to load subset channels. For FreshMotor we don't have subsets yet as we need all channels. Thus we do construct subset_channels manually and make it equal to all channels.
    if DATASET_NAME == "BallSqueezingHD_modified":
        with open(f"results/subset/{DATASET_NAME}/subsets_data.pkl", "rb") as f:
            subsets_data = pickle.load(f)
        subset_channels = subsets_data[subset_type]["all"]
        
    elif DATASET_NAME == "FreshMotor":
        # make subset_channels equal to all channels
        filename = files[0] # select one
        rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
        all_channels = rec['amp']['channel'].values.tolist()
        subset_channels = all_channels

# motor_sampling is the approach we use to subsample channels based on motor area locations. See src/subset/sparsify_chs_from_sens.ipynb
motor_sampling = False
if motor_sampling:
    if DATASET_NAME == "BS_Laura":
        subset_channels = np.load(sparsified_data_path, allow_pickle=True)
        
no_subsampling = True
if no_subsampling:
     # make subset_channels equal to all channels
    filename = files[0] # select one
    print(filename)
    rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
    all_channels = rec['amp']['channel'].values.tolist()
    subset_channels = all_channels
    
print(len(subset_channels))

datasets/raw/vfc_hd/sub-01/nirs/sub-01_ses-02_task-WordStroop_run-01_nirs.snirf
214


In [12]:
filename = files[0] # select one
rec = cedalion.io.read_snirf(filename)[0]  # read snirf files

# subset the data
rec['amp'] = rec['amp'].sel(channel=subset_channels)

# subset measurement list
meas_list = rec._measurement_lists["amp"]
meas_list = meas_list[meas_list["channel"].isin(subset_channels)].reset_index(drop=True)

# both should then be equal
print(f'Number of channels in rec["amp"]: {len(set(rec["amp"].channel.values))}')
print(f'Number of channels in meas_list: {len(set(meas_list.channel.values))}')

head_icbm152 = dot.get_standard_headmodel('icbm152')  


if DATASET_NAME == "BS_Laura":
    # this is required for BU Data (Laura's)
    T = np.array([
        [-9.57882733e-01, -7.20806358e-03,  6.20193531e-03, 2.21208571e+02],
        [-2.02271710e-02,  6.03819925e-02,  9.94046165e-01, -2.03010603e+01],
        [-8.79481533e-03, -1.02761992e+00,  6.59199998e-02, 2.87749135e+02],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 1.00000000e+00]])
    
    ninja_aligned = rec.geo3d.points.apply_transform(T)
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(ninja_aligned)
else:
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(rec.geo3d)
    

fwm = cedalion.dot.forward_model.ForwardModel(
    head_icbm152, 
    geo3d_snapped_ijk,
    meas_list
)

fluence_fname = os.path.join(pre_processed_path, "fluence_" + DATASET_NAME + ".h5")
sensitivity_fname = os.path.join(pre_processed_path, "sensitivity_" + DATASET_NAME + ".h5")

# compute fluence and sensitivity only once
# fwm.compute_fluence_mcx(fluence_fname)
# fwm.compute_sensitivity(fluence_fname, sensitivity_fname)

Adot = load_Adot(sensitivity_fname)

recon = None  # NEW: recon is now created per-recording/per-view in the main loop


Number of channels in rec["amp"]: 214
Number of channels in meas_list: 214


In [13]:
import cedalion.vis.blocks as vbx
import pyvista as pv
import cedalion.dataclasses as cdc
# keep only points that are not of type "landmark", i.e. source and detector points
geo3d_snapped_ijk = geo3d_snapped_ijk[geo3d_snapped_ijk.type != cdc.PointType.LANDMARK]

# now we plot the head same as before...
plt = pv.Plotter()
vbx.plot_surface(plt, head_icbm152.brain, color="#d3a6a1")
vbx.plot_surface(plt, head_icbm152.scalp, opacity=.1)
# but use the plot_labeled_points() function to add the snapped geo3d.
# The flag "show_labels" can be used to show the source, detector, and landmark names
vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=True)
# plt.show()
# save image
plt.screenshot("head_with_snapped_points_" + DATASET_NAME + ".png")

2026-06-21 14:10:10.531 (  23.644s) [    7FB5B1642740]vtkXOpenGLRenderWindow.:1416  WARN| bad X server connection. DISPLAY=:99.0


pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [14]:
import cedalion.vis.blocks as vbx
from cedalion.io.forward_model import FluenceFile

# pull fluence values from the corresponding source and detector pair

with FluenceFile(fluence_fname) as fluence_file:
    f = fluence_file.get_fluence("S12", 760) * fluence_file.get_fluence("D19", 760)

f = np.log10(np.clip(f, min=f[f > 0].min()))
vf = pv.wrap(f)

plt = pv.Plotter()

plt.add_volume(
    vf,
    log_scale=False,
    cmap="plasma_r",
    clim=(-10, -0),
    scalar_bar_args={
        "title": r"$log_{10}("
        r"F(\vec{x}_{src},\vec{x}) * F(\vec{x}, \vec{x}_{det})"
        ")$"
    },
)
vbx.plot_surface(plt, head_icbm152.brain, color="w")
vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=False)
vbx.camera_at_cog(plt, head_icbm152.brain, rpos=[300, 150, 150])
# plt.show()
plt.screenshot(f"fluence_visualization_{DATASET_NAME}.png")

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [15]:
import cedalion.vis.anatomy.sensitivity_matrix as sensitivity_matrix

plotter = sensitivity_matrix.Main(
    sensitivity=Adot,
    brain_surface=head_icbm152.brain,
    head_surface=head_icbm152.scalp,
    labeled_points=geo3d_snapped_ijk,
)
plotter.plot(high_th=0, low_th=-3)
# plotter.plt.show()
# save the figure
plotter.plt.screenshot(f"sensitivity_visualization_{DATASET_NAME}.png")

pyvista_ndarray([[[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 ...,

                 [[255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255],
                  ...,
                  [255, 255, 255],
                  [255, 255, 255],
                  [255, 255, 255]],

                 [[255, 255, 255],
  

In [16]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/vfc_hd/full')

In [17]:
# NEW: recon debug check updated for per-view creation flow
print(len(meas_list['channel']))
print(rec["amp"].channel.size)
if recon is None:
    print("recon is configured per recording/view in the main loop")
else:
    print(recon._F.shape)


428
214
recon is configured per recording/view in the main loop


In [18]:
stim = cedalion.io.read_events_from_tsv(files[0].replace('nirs.snirf', 'events.tsv'))

stim, rec = standardize_trial_types(DATASET_NAME, files[0], stim, rec)
rec.stim.head()

,onset,duration,value,trial_type
0,18.60,18.0,1.0,WordCongruent
1,50.82,18.0,1.0,WordIncongruent
2,84.17,18.0,1.0,WordIncongruent
3,113.06,18.0,1.0,WordIncongruent
4,139.76,18.0,1.0,WordCongruent


In [19]:
print(np.diff(np.sort(rec.stim.onset.values)))
print(np.min(np.diff(np.sort(rec.stim.onset.values))))

[32.22 33.35 28.89 26.7  27.57 35.   25.17 33.21 28.62 32.81 33.59]
25.169999999999987


In [20]:
# JOB-ARRAY HANDOFF: heavy image-recon preprocessing now runs outside the notebook.
# 1) Create the file list:
#    python src/subset/make_recon_param_job_list.py --dataset BS_Laura --subset full
#    This writes recon_param_files.txt inside pre_processed_path.
#
# 2) Submit the array job after checking the printed job_array range.
#    The %10 cap means at most 10 array tasks run at the same time:
#    sbatch \
#   --export=ALL,DATASET=vfc_hd,SUBSET=full \
#   --array=0-16%10 \
#   scripts/sbatch_recon_param_preprocessing.sh
#
# 3) After all jobs finish, run this cell to load per-recording metadata.

metadata_dir = pre_processed_path / "_job_metadata"
if not metadata_dir.exists():
    raise FileNotFoundError(
        f"Missing metadata directory: {metadata_dir}. Run the SLURM preprocessing jobs first."
    )

file_sens_drop_parcels_dict = {}
skipped_subjects = []
job_metadata = []

for meta_file in sorted(metadata_dir.glob("*.pkl")):
    with open(meta_file, "rb") as handle:
        meta = pickle.load(handle)
    job_metadata.append(meta)

    if meta.get("status") == "ok":
        file_sens_drop_parcels_dict[meta["file"]] = {
            "sensitive_parcels": meta["sensitive_parcels"],
            "dropped_parcels": meta["dropped_parcels"],
        }
    else:
        skipped_subjects.append(meta["file"])

with open(pre_processed_path / "file_sens_drop_parcels_dict.pkl", "wb") as handle:
    pickle.dump(file_sens_drop_parcels_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

expected_jobs = len(files)
finished_jobs = len(job_metadata)
print(f"metadata files loaded: {finished_jobs} / expected raw files: {expected_jobs}")
print(f"processed recordings: {len(file_sens_drop_parcels_dict)}")
print(f"skipped recordings: {len(skipped_subjects)}")

if finished_jobs != expected_jobs:
    missing = expected_jobs - finished_jobs
    print(f"WARNING: metadata count does not match expected raw files. Missing jobs: {missing}")

if skipped_subjects:
    print("Skipped files:")
    for f in skipped_subjects:
        print(f"  {f}")


metadata files loaded: 17 / expected raw files: 17
processed recordings: 16
skipped recordings: 1
Skipped files:
  datasets/raw/vfc_hd/sub-13/nirs/sub-13_ses-01_task-WordStroop_run-01_nirs.snirf


In [21]:
# JOB-ARRAY HANDOFF: file_sens_drop_parcels_dict is saved in the metadata reload cell above.
print(pre_processed_path / "file_sens_drop_parcels_dict.pkl")



datasets/pre_processed/imageRecon_params/vfc_hd/full/file_sens_drop_parcels_dict.pkl


In [22]:
# Load the common parcel template used for segmentation.
if DATASET_NAME == "BS_Laura":
    template_dataset_name = "BallSqueezingHD_modified" 
else:
    template_dataset_name = DATASET_NAME
    
sens_parcel_template_path = Path(f"datasets/parcel_templates/parcel_template_{template_dataset_name}.pkl")

if not sens_parcel_template_path.exists():
    raise FileNotFoundError(f"Could not find parcel template: {sens_parcel_template_path}")

with open(sens_parcel_template_path, "rb") as handle:
    template_sens_parcel_list = pickle.load(handle)

print("template:", sens_parcel_template_path)
print("template parcels:", len(template_sens_parcel_list))

# For each processed recording, check how many sensitive parcels are contained in the common template.
for f, sens_drop_info in file_sens_drop_parcels_dict.items():
    sensitive_parcels = sens_drop_info["sensitive_parcels"]
    dropped_parcels = sens_drop_info["dropped_parcels"]

    num_sensitive_in_template = sum(1 for parcel in sensitive_parcels if parcel in template_sens_parcel_list)
    num_dropped_in_template = sum(1 for parcel in dropped_parcels if parcel in template_sens_parcel_list)

    print(f"  Sensitive parcels in template: {num_sensitive_in_template} / {len(template_sens_parcel_list)}")


template: datasets/parcel_templates/parcel_template_vfc_hd.pkl
template parcels: 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143
  Sensitive parcels in template: 143 / 143


In [23]:
Adot.shape

(214, 35018, 2)

In [24]:
# import matplotlib.pyplot as plt
# # Plot as barplot the number of sensitive parcels in the template for each subject
# files = list(file_sens_drop_parcels_dict.keys())
# num_sensitive_in_template_list = []
# for f in files:
#     sensitive_parcels = file_sens_drop_parcels_dict[f]['sensitive_parcels']
#     num_sensitive_in_template = len(set(sensitive_parcels) & set(template_sens_parcel_list))
#     num_sensitive_in_template_list.append(num_sensitive_in_template)
# plt.figure(figsize=(10, 6))
# plt.bar(range(len(files)), num_sensitive_in_template_list, color='blue')
# # print the number of sensitive parcels in the template for each subject on top of the bars
# for i, num in enumerate(num_sensitive_in_template_list):    
#     plt.text(i, num + 0.5, str(num), ha='center', va='bottom')
# # plt.plot(range(len(files)), num_sensitive_in_template_list, color='red')
# plt.xticks(range(len(files)), [f.split('/')[3].replace('.snirf', '') for f in files], rotation=45)
# plt.xlabel('Subjects')
# plt.ylabel('Number of Sensitive Parcels in Template')
# plt.ylim(0, len(template_sens_parcel_list) + 10)
# plt.axhline(y=len(template_sens_parcel_list), color='gray', linestyle='dashed')
# # plt.legend()
# # plt.title('Sensitive Parcels in Template for Each Subject')
# plt.tight_layout()
# plt.show()

In [25]:
print("raw files:", len(files))
print("processed recordings from metadata:", len(file_sens_drop_parcels_dict))
print("skipped recordings:", len(skipped_subjects))


raw files: 17
processed recordings from metadata: 16
skipped recordings: 1


In [26]:
# JOB-ARRAY HANDOFF: rec["amp_clean"] is produced inside each external job and is not kept in notebook memory.
# Load saved .pkl files below for inspection instead.


In [27]:
# JOB-ARRAY HANDOFF: standardized event tables are stored in each saved .pkl as data["rec_stim"].
# Example: run the next data-loading cell, then inspect data["rec_stim"].


In [28]:
# JOB-ARRAY HANDOFF: use pre_processed_path-based glob patterns instead of hard-coded paths.


In [29]:
# NEW: load any augmented view for the first available preprocessed recording
pattern = str(pre_processed_path / "am_1__as_1" / "sub-*" / "*.pkl")
matched = sorted(glob.glob(pattern))
if not matched:
    raise FileNotFoundError(f"No files matched pattern: {pattern}")
with open(matched[0], "rb") as handle:
    data = pickle.load(handle)
print("loading:", matched[0])
data["rec_stim"].head()


loading: datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl


,onset,duration,value,trial_type
0,18.60,18.0,1.0,WordCongruent
1,50.82,18.0,1.0,WordIncongruent
2,84.17,18.0,1.0,WordIncongruent
3,113.06,18.0,1.0,WordIncongruent
4,139.76,18.0,1.0,WordCongruent


In [30]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/vfc_hd/full')

In [31]:
# load data for visualization
if DATASET_NAME == "BallSqueezingHD_modified":
    subject_name = 'sub-185'
    base_name = f'{subject_name}_task-BallSqueezing_run-3_nirs'

elif DATASET_NAME == "FreshMotor":
    subject_name = 'sub-01'
    base_name = f'{subject_name}_task-FRESHMOTOR_run-left2s_nirs'

elif DATASET_NAME == "BS_Laura":
    subject_name = 'sub-583'
    subject_name = 'sub-633'
    base_name = f'{subject_name}_task-BS_run-01_nirs'
elif DATASET_NAME == "vfc_hd":
    subject_name = 'sub-12'
    base_name = f'{subject_name}_ses-01_task-WordStroop_run-01_nirs'
elif DATASET_NAME == "Anderson_sparse":
    subject_name = 'sub-1'
    base_name = f'{subject_name}_ses-1_task-WordStroop_run-1_nirs'

pattern = str(pre_processed_path / "am_1__as_1" / subject_name / f"{base_name}.pkl")
matched = sorted(glob.glob(pattern))
if not matched:
    raise FileNotFoundError(f"No files matched pattern: {pattern}")

file_to_plot = matched[0]
print("loading:", file_to_plot)

# load data
with open(file_to_plot, 'rb') as handle:
    data = pickle.load(handle)


loading: datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-12/sub-12_ses-01_task-WordStroop_run-01_nirs.pkl


In [32]:
data['delta_conc']

<xarray.DataArray (time: 9529, parcel: 601, chromo: 2)> Size: 92MB
array([[[ 0.00000000e+00,  0.00000000e+00],
        [ 5.92218566e-06, -3.89008146e-06],
        [ 2.02532589e-06, -2.35272169e-06],
        ...,
        [ 1.02684186e-11,  7.06618540e-13],
        [-1.82838618e-11, -3.44789710e-11],
        [ 4.14087566e-11,  1.93764596e-11]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [ 5.68962467e-06, -3.92079618e-06],
        [ 1.89298470e-06, -2.35874524e-06],
        ...,
        [ 1.04992264e-11,  5.85905991e-13],
        [-1.79209432e-11, -3.42118544e-11],
        [ 4.23441990e-11,  1.84556752e-11]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [ 5.50657381e-06, -3.97276816e-06],
        [ 1.78759979e-06, -2.37595642e-06],
        ...,
...
        ...,
        [-2.07717268e-12,  3.33141893e-12],
        [ 1.39505476e-10,  2.42982545e-11],
        [-5.67610576e-12,  4.25893892e-12]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [-2.94143655e-06,  4.93612491e-06],
        [-1.37086551e-06,  2.26986449e-06],
        ...,
        [-1.90982377e-12,  3.29989415e-12],
        [ 1.38657232e-10,  2.43395626e-11],
        [-5.65468893e-12,  4.45488238e-12]],

       [[ 0.00000000e+00,  0.00000000e+00],
        [-2.77999277e-06,  4.82395622e-06],
        [-1.28904417e-06,  2.20263115e-06],
        ...,
        [-1.71637978e-12,  3.24190793e-12],
        [ 1.37836690e-10,  2.44693597e-11],
        [-5.58736971e-12,  4.41611564e-12]]], shape=(9529, 601, 2))
Coordinates:
  * chromo   (chromo) <U3 24B 'HbO' 'HbR'
  * time     (time) float64 76kB 21.16 21.22 21.27 21.33 ... 567.4 567.5 567.5
    samples  (time) int64 76kB 369 370 371 372 373 ... 9893 9894 9895 9896 9897
  * parcel   (parcel) object 5kB 'Background+FreeSurfer_Defined_Medial_Wall' ...

In [33]:
data.keys()

dict_keys(['conc_pcr', 'delta_conc', 'rec_stim', 'sensitive_parcels', 'view_id', 'alpha_meas', 'alpha_spatial', 'alpha_meas_0', 'alpha_spatial_0', 'alpha_meas_multiplier', 'alpha_spatial_multiplier', 'k_alpha_meas'])

<!-- # Calculate block averages in optical density
 -->


In [34]:
data['conc_pcr']

Magnitude,[[[0.13485220453720206 0.20808881554483202 0.17775571238491367 ... -0.3030141499215587 -0.20342586912715627 -0.10375909031069194] [0.06777197550559153 0.07936838301441002 0.060959984659729335 ... -0.2118730759598505 -0.16917265857286584 -0.11007025694265773] [0.1298983873903884 0.1864190815764117 0.1592724958540799 ... -0.11261135151353299 -0.06198304962430737 -0.02140404906735246] ... [0.386812474385909 0.35232327911486605 0.33209316771250397 ... -0.13616657401581023 -0.17465091773974636 -0.11532320732340814] [0.35846741906764656 0.37126131834915677 0.3201876618221967 ... -0.11870650917362889 -0.1348499947062565 -0.14949496236193094] [13.115208178419316 36.56172409507815 5.83986587047688 ... -60.706363754299566 -64.9695505532752 -18.258831989670654]] [[0.03466018087125868 0.05326615025444273 0.06194664352458604 ... -0.06415657484724432 -0.04955186632071012 -0.027884663639077795] [-0.00310298702504493 -0.0013197447665389246 -0.0031732722663677885 ... 0.07832921238390864 0.053684405417127753 0.04600736473486198] [0.025483682411579184 0.022738884576640736 0.04596801767245211 ... -0.04406391403641517 -0.027352677719421358 -0.00937282088654614] ... [-0.10469943146572547 -0.07166352928892743 -0.09322944226654502 ... 0.05528306547390528 0.12398263105299097 0.05944346239836995] [-0.15154304306398397 -0.12072538848745382 -0.1291545751163029 ... -0.011163216129086773 0.01724494410234495 -0.008036372130442115] [-15.914387423825517 -51.32115668605164 -40.53995785564251 ... -5.427948851650168 -0.43085322153247585 -18.664004343870754]]]
Units,micromolar


In [35]:
# # Recompute od_pcr1 for one recording only, for visualization functions that need rec["od_pcr1"]

# from cedalion.io import read_events_from_tsv

# raw_file_for_viz = files[0]  # or set manually

# records = cedalion.io.read_snirf(raw_file_for_viz)
# rec = records[0]

# rec["amp"] = rec["amp"].sel(channel=subset_channels)

# stim = read_events_from_tsv(raw_file_for_viz.replace("nirs.snirf", "events.tsv"))
# rec.stim = rec.stim.sort_values(by="onset")
# stim, rec = standardize_trial_types(DATASET_NAME, raw_file_for_viz, stim, rec)

# rec["rep_amp"] = quality.repair_amp(rec["amp"], median_len=3, method="linear")
# rec["od_amp"], baseline = nirs.cw.int2od(rec["rep_amp"], return_baseline=True)

# rec["od_tddr"] = motion_correct.tddr(rec["od_amp"])
# rec["od_tddr_wavel"] = motion_correct.wavelet(rec["od_tddr"])

# rec["od_hpfilt"] = rec["od_tddr_wavel"].cd.freq_filter(
#     fmin=0.008,
#     fmax=0,
#     butter_order=4,
# )

# rec["amp_clean"] = cedalion.nirs.cw.od2int(rec["od_hpfilt"], baseline)

# ch_preproc = {
#     "sci_thresh": 0.5,
#     "psp_thresh": 0.1,
#     "window_len": 5 * units.s,
#     "dark_sat_thresh": [1e-3, 0.84],
#     "perc_time_clean": 0.5,
# }

# list_bad_ch = get_bad_ch_mask(rec["amp_clean"], ch_preproc)

# dpf = xr.DataArray(
#     [6, 6],
#     dims="wavelength",
#     coords={"wavelength": rec["amp"].wavelength},
# )

# rec["conc"] = cedalion.nirs.cw.od2conc(
#     rec["od_hpfilt"],
#     rec.geo3d,
#     dpf,
#     spectrum="prahl",
# )

# chromo_var = quality.measurement_variance(
#     rec["conc"],
#     list_bad_channels=list_bad_ch,
#     bad_rel_var=1e6,
#     calc_covariance=False,
# )

# rec["conc_pcr"], gb_comp_rem = physio.global_component_subtract(
#     rec["conc"],
#     ts_weights=1 / chromo_var,
#     k=0,
#     spatial_dim="channel",
#     spectral_dim="chromo",
# )

# rec["od_pcr1"] = cedalion.nirs.cw.conc2od(
#     rec["conc_pcr"],
#     rec.geo3d,
#     dpf,
#     spectrum="prahl",
# )

# print("ready:", raw_file_for_viz)
# print(rec["od_pcr1"])

<!-- Blockaverage od_pcr1 -->

In [36]:
# # segment data into epochs
# if DATASET_NAME in ["BallSqueezingHD_modified", "BS_Laura"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["right"],  # select left/right events
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_od_pcr1 = epochs_blcorrected.groupby("trial_type").mean("epoch")

In [37]:
from IPython.display import Image
def display_image(fname : str):
    display(Image(data=open(fname,'rb').read(), format='png'))

In [38]:
results_path_prefix = f'results/{subset_type}/{DATASET_NAME}/{subject_name}'
os.makedirs(results_path_prefix, exist_ok=True)

In [39]:
# # Plot block averages. Please ignore errors if the plot is too small in the HD case

# filename = f"results/{subset_type}/{DATASET_NAME}/blockaverage_channel_space_{subset_type}.png"

# # noPlts2 = int(np.ceil(np.sqrt(len(blockaverage_od_pcr1.channel))))
# # f,ax = plt.subplots(noPlts2,noPlts2, figsize=(12,10))
# # ax = ax.flatten()
# # for i_ch, ch in enumerate(blockaverage_od_pcr1.channel):
# #     for ls, trial_type in zip(["-", "--"], blockaverage_od_pcr1.trial_type):
# #         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=760, trial_type=trial_type, channel=ch), "r", lw=2, ls=ls)
# #         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=850, trial_type=trial_type, channel=ch), "b", lw=2, ls=ls)

# #     ax[i_ch].grid(1)
# #     ax[i_ch].set_title(ch.values)
# #     ax[i_ch].set_ylim(-.02, .02)
# #     ax[i_ch].set_axis_off()
# #     ax[i_ch].axhline(0, c="k")
# #     ax[i_ch].axvline(0, c="k")

# # # plt.suptitle("760nm: r | 850nm: b | left: - | right: --")
# # plt.suptitle("HbO: r | HbR: b | left: - | right: --")

# # plt.tight_layout()
# # plt.savefig(filename)

In [40]:
def match_landmark_labels(rec):
    subject_nasion_mask = rec.geo3d['label'].data == 'NASION'

    new_labels = rec.geo3d['label'].data.copy()
    new_labels[subject_nasion_mask] = 'Nz'

    # Create new geo3d with updated labels
    rec.geo3d = rec.geo3d.assign_coords(label=new_labels)
    # print(rec.geo3d['label'].data)

    return rec

In [42]:
# rec = match_landmark_labels(rec)

In [43]:
# # Viz reconstruction on Channel Space
# import cedalion.vis.anatomy
# filename_scalp = f"results/{subset_type}/{DATASET_NAME}/scalp_plot_ts_{subset_type}.png"

# data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="right")
# # data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="WordCongruent")
# # scalp_plot_gif expects the time dimension to be named 'time'
# data_ts = data_ts.rename({"reltime": "time"})

# # call plot function 
# cedalion.vis.anatomy.scalp_plot_gif(
#     data_ts,
#     rec.geo3d,
#     filename=filename_scalp,
#     time_range=(-5, 11, 0.5) * units.s,
#     scl=(-0.01, 0.01),
#     fps=6,
#     optode_size=6,
#     optode_labels=True,
#     str_title="OD 850 nm",
# )
# display_image(f"{filename_scalp}.gif")

<!-- Blockaverage_delta_conc -->

In [44]:
# if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         data['rec_stim'],  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )    

# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_delta_conc = epochs_blcorrected.groupby("trial_type").mean("epoch")
# # blockaverage_delta_conc

In [45]:
# vertex_parcels = head_icbm152.brain.vertex_coords['parcel']
# vertex_parcels = np.array(vertex_parcels)

# parcel_index = blockaverage_delta_conc.get_index("parcel")  # pandas index
# vertex_parcel_idx = parcel_index.get_indexer(vertex_parcels)

# parcel_data = blockaverage_delta_conc.values  # shape (2, 107, 601, 2)

# # Broadcast using integer indexing on axis=2 (parcel axis)
# vertex_activity = parcel_data[:, :, vertex_parcel_idx, :]

# n_vertices = len(vertex_parcel_idx)

# vertex_da = xr.DataArray(
#     vertex_activity,
#     dims=("trial_type", "reltime", "vertex", "chromo"),
#     coords=dict(
#         trial_type=blockaverage_delta_conc.trial_type,
#         reltime=blockaverage_delta_conc.reltime,
#         chromo=blockaverage_delta_conc.chromo,

#         # vertex index
#         vertex=np.arange(n_vertices),

#         # parcel label of each vertex
#         parcel_of_vertex=("vertex", vertex_parcels),

#         # NEW: is_brain flag
#         is_brain=("vertex", np.ones(n_vertices, dtype=bool))
#     )
# )

# # vertex_da

In [46]:
subset_type

'full'

In [47]:
# filename_multiview = f'{results_path_prefix}/image_recon_multiview_{subset_type}'

# # prepare data
# # X_ts = vertex_da.sel(trial_type="right").rename({"reltime": "time"})
# X_ts = vertex_da.sel(trial_type="WordCongruent").rename({"reltime": "time"})
# X_ts = X_ts.transpose("vertex", "chromo", "time")

# # t_plot = 5.0
# # X_frame = X_ts.sel(time=t_plot, method="nearest")

# scl = np.percentile(np.abs(X_ts.sel(chromo='HbO')).pint.dequantify(), 99)
# clim = (-scl,scl)

# cedalion.vis.anatomy.image_recon_multi_view(
# # cedalion.vis.anatomy.image_recon_view(
#     X_ts,  # time series data; can be 2D (static) or 3D (dynamic)
#     # X_frame,
#     head_icbm152,
#     cmap='seismic',
#     clim=clim,
#     view_type='hbo_brain',
#     title_str='HbO / µM',
#     filename=filename_multiview,
#     SAVE=True,
#     time_range=(-2,10,0.5)*units.s,
#     fps=5,
#     geo3d_plot = None, #  geo3d_plot
#     wdw_size = (1024, 768)
# )
# display_image(filename_multiview+'.gif')

In [48]:
# import pickle
# from pathlib import Path

# import numpy as np
# import matplotlib.pyplot as plt
# import xarray as xr

# # Required variables from previous notebook cells:
# # subject_name
# # base_name
# # pre_processed_path
# # results_path_prefix
# # head_icbm152
# # get_param_config_folder

# alpha_meas_grid = [0.1, 1.0, 10.0]
# alpha_spatial_grid = [0.1, 1.0, 10.0]

# plot_chromo = "HbO"
# t_plot = 7.0  # absolute time in seconds from the saved delta_conc time axis

# out_dir = Path(results_path_prefix) / "image_recon_view_3x3_pngs"
# out_dir.mkdir(parents=True, exist_ok=True)

# vertex_parcels = np.asarray(head_icbm152.brain.vertex_coords["parcel"])
# brain_vertex = np.arange(len(vertex_parcels))

# png_files = {}
# grid_meta = {}
# all_vals = []

# # First pass: load all configs, map parcel data to brain vertices, collect color scale
# vertex_frames = {}

# for am_mult in alpha_meas_grid:
#     for as_mult in alpha_spatial_grid:
#         config_folder = get_param_config_folder(am_mult, as_mult)

#         file_to_load = (
#             pre_processed_path
#             / config_folder
#             / subject_name
#             / f"{base_name}.pkl"
#         )

#         if not file_to_load.exists():
#             raise FileNotFoundError(f"Missing file: {file_to_load}")

#         with open(file_to_load, "rb") as handle:
#             data_grid = pickle.load(handle)

#         delta = data_grid["delta_conc"]

#         # Select one time point, but keep both HbO and HbR because image_recon_view expects chromo.
#         snap = delta.sel(time=t_plot, method="nearest")

#         parcel_index = snap.get_index("parcel")
#         vertex_parcel_idx = parcel_index.get_indexer(vertex_parcels)

#         vertex_values = np.full(
#             (len(vertex_parcels), len(snap.chromo)),
#             np.nan,
#             dtype=float,
#         )

#         valid = vertex_parcel_idx >= 0
#         vertex_values[valid, :] = snap.values[vertex_parcel_idx[valid], :]

#         X_frame = xr.DataArray(
#             vertex_values,
#             dims=("vertex", "chromo"),
#             coords={
#                 "vertex": brain_vertex,
#                 "chromo": snap.chromo.values,
#                 "parcel_of_vertex": ("vertex", vertex_parcels),
#                 "is_brain": ("vertex", np.ones(len(vertex_parcels), dtype=bool)),
#             },
#         )

#         vertex_frames[(am_mult, as_mult)] = X_frame

#         vals = X_frame.sel(chromo=plot_chromo).values
#         all_vals.append(vals[np.isfinite(vals)])

#         grid_meta[(am_mult, as_mult)] = {
#             "alpha_meas": data_grid.get("alpha_meas"),
#             "alpha_spatial": data_grid.get("alpha_spatial"),
#         }

# all_vals = np.concatenate(all_vals)
# scl = np.percentile(np.abs(all_vals), 99)

# if not np.isfinite(scl) or scl == 0:
#     scl = 1.0

# clim = (-scl, scl)

# # Second pass: save one superior-view PNG per config using image_recon_view()
# for am_mult in alpha_meas_grid:
#     for as_mult in alpha_spatial_grid:
#         X_frame = vertex_frames[(am_mult, as_mult)]

#         png_base = out_dir / f"superior_am_{am_mult:g}__as_{as_mult:g}"

#         cedalion.vis.anatomy.image_recon_view(
#             X_frame,
#             head_icbm152,
#             cmap="seismic",
#             clim=clim,
#             view_type="hbo_brain",
#             view_position="superior",
#             title_str=f"{plot_chromo} / uM",
#             filename=str(png_base),
#             SAVE=True,
#             geo3d_plot=None,
#             wdw_size=(700, 700),
#         )

#         png_files[(am_mult, as_mult)] = Path(str(png_base) + ".png")

# # Third pass: compose the 9 saved PNGs into one matplotlib figure
# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# for row, am_mult in enumerate(alpha_meas_grid):
#     for col, as_mult in enumerate(alpha_spatial_grid):
#         ax = axes[row, col]
#         png_file = png_files[(am_mult, as_mult)]
#         meta = grid_meta[(am_mult, as_mult)]

#         img = plt.imread(png_file)
#         ax.imshow(img)
#         ax.axis("off")

#         ax.set_title(
#             f"am x{am_mult:g}, as x{as_mult:g}\n"
#             f"am={meta['alpha_meas']:.2e}, as={meta['alpha_spatial']:.2e}",
#             fontsize=8,
#         )

# fig.suptitle(
#     f"Superior view using image_recon_view | "
#     f"{plot_chromo} @ t={float(t_plot):.2f}s | "
#     f"{subject_name} | {base_name}",
#     fontsize=11,
# )

# fig.tight_layout()

# combined_png = Path(results_path_prefix) / f"image_recon_view_superior_3x3_t{t_plot:g}.png"
# fig.savefig(combined_png, dpi=200, bbox_inches="tight")

# plt.show()

# print("Saved combined figure:", combined_png)
# print("Saved individual PNGs in:", out_dir)

# Segmentation

In [49]:
pre_processed_path

PosixPath('datasets/pre_processed/imageRecon_params/vfc_hd/full')

In [50]:
# load (all) parcel files
preproc_files_path = str(pre_processed_path / 'am_*__as_*' / 'sub-*' / '*.pkl')
proc_pkl_files = glob.glob(preproc_files_path)

len(proc_pkl_files), proc_pkl_files[:2]

(144,
 ['datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_0.1/sub-08/sub-08_ses-01_task-WordStroop_run-01_nirs.pkl',
  'datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_0.1/sub-26/sub-26_ses-01_task-WordStroop_run-01_nirs.pkl'])

In [52]:
use_channel_space = True
if use_channel_space:
    processed_path = Path(f'datasets/processed/{augmentation_strategy}/{DATASET_NAME}/"channel_space_data"/{subset_type}')
else:    
    processed_path = Path(f'datasets/processed/{augmentation_strategy}/{DATASET_NAME}/{subset_type}')
processed_path.mkdir(parents=True, exist_ok=True)
processed_path

PosixPath('datasets/processed/imageRecon_params/vfc_hd/"channel_space_data"/full')

<!-- ### Create template sensitive parcels -->

In [53]:
create_orload_parcel_template = "load"  # "load" or "create"

if create_orload_parcel_template == "create":
    # Load any file as all processed files have the same sensitive parcels
    # f = f"datasets/{subset_type}_pre_processed/BallSqueezingHD_modified/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"
    
    if DATASET_NAME == "BallSqueezingHD_modified":
        f = sorted(glob.glob("datasets/pre_processed/BallSqueezingHD_modified/*/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"))[0]
    elif DATASET_NAME == "vfc_hd":
        # f = sorted(glob.glob("datasets/pre_processed/vfc_hd/*/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl"))[0]
        f = sorted(glob.glob("datasets/pre_processed/imageRecon_params/vfc_hd/full/am_1__as_1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl"))[0]
    elif DATASET_NAME == "Anderson_sparse":
        f = sorted(glob.glob("datasets/pre_processed/Anderson_sparse/*/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs.pkl"))[0]
        
    with open(f, 'rb') as handle:
        data = pickle.load(handle)
        
    print(len(data['sensitive_parcels']))

    # save the loaded sensitive parcels as template as pkl file
    folder_path = 'datasets/parcel_templates'
    os.makedirs(folder_path, exist_ok=True)
    # sens_parcel_template_path = os.path.join(folder_path, f'parcel_template_{DATASET_NAME}.pkl')
    # sens_parcel_template_path = os.path.join(folder_path, f'{subset_type}_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
    sens_parcel_template_path = "datasets/parcel_templates/parcel_template_vfc_hd.pkl"
    # sens_parcel_template_path = "datasets/parcel_templates/parcel_template_Anderson_sparse.pkl"

    with open(sens_parcel_template_path, 'wb') as handle:
        pickle.dump(data['sensitive_parcels'], handle)
    
    print(f"Parcel template saved to {sens_parcel_template_path}")
        
        
elif create_orload_parcel_template == "load":
    folder_path = 'datasets/parcel_templates'
    
    if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
        # sens_parcel_template_path = os.path.join(folder_path, 'subset_2_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
        sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_BallSqueezingHD_modified.pkl')
    elif DATASET_NAME == "vfc_hd" or DATASET_NAME == "Anderson_sparse":
        sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_vfc_hd.pkl')
        # sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_Anderson_sparse.pkl')
    
    with open(sens_parcel_template_path, 'rb') as handle:
        template_sens_parcel_list = pickle.load(handle)
        
    print(len(template_sens_parcel_list))
    print(f"Parcel template loaded from {sens_parcel_template_path}")

143
Parcel template loaded from datasets/parcel_templates/parcel_template_vfc_hd.pkl


In [54]:
# check how many of the sensitive parcels are motor are
motor_labels = [l for l in template_sens_parcel_list if l.startswith("SomMot")]
print(f"{len(motor_labels)} motor labels")

28 motor labels


In [55]:
template_sens_parcel_list

['ContA_IPS_2_LH',
 'ContA_IPS_3_RH',
 'ContA_PFCl_1_LH',
 'ContA_PFCl_1_RH',
 'ContA_PFCl_2_LH',
 'ContA_PFCl_2_RH',
 'ContA_PFCl_3_RH',
 'ContA_PFCl_4_LH',
 'ContA_PFCl_5_RH',
 'ContA_PFCl_6_LH',
 'ContA_PFClv_1_LH',
 'ContA_PFClv_2_LH',
 'ContB_PFCd_1_LH',
 'ContB_PFCl_1_LH',
 'ContB_PFCld_1_RH',
 'ContB_PFCld_2_RH',
 'ContB_PFCld_4_RH',
 'ContB_PFCld_5_RH',
 'ContB_PFCld_7_RH',
 'ContB_PFClv_1_LH',
 'ContB_PFClv_2_LH',
 'ContB_PFClv_2_RH',
 'ContB_PFClv_3_LH',
 'ContB_PFClv_3_RH',
 'ContB_PFClv_4_RH',
 'ContB_PFClv_5_RH',
 'ContB_PFClv_6_RH',
 'ContB_PFCmp_1_RH',
 'ContB_Temp_1_LH',
 'ContB_Temp_2_RH',
 'DefaultA_PFCd_1_LH',
 'DefaultA_PFCd_1_RH',
 'DefaultA_PFCd_2_LH',
 'DefaultA_PFCd_2_RH',
 'DefaultA_PFCd_3_LH',
 'DefaultA_PFCd_3_RH',
 'DefaultA_PFCm_1_LH',
 'DefaultA_PFCm_3_LH',
 'DefaultA_PFCm_4_RH',
 'DefaultA_PFCm_5_LH',
 'DefaultA_PFCm_5_RH',
 'DefaultA_PFCm_6_LH',
 'DefaultA_PFCm_7_RH',
 'DefaultA_Temp_1_RH',
 'DefaultB_AntTemp_3_RH',
 'DefaultB_PFCd_1_LH',
 'DefaultB_PFCd

In [56]:
min_len_sens_parcels = float('inf')
min_len_i = None

for i in range(len(proc_pkl_files)):
    with open(proc_pkl_files[i], 'rb') as handle:
        data_pickle = pickle.load(handle)
        delta_brain = data_pickle['delta_conc']
        sensitive_parcels = data_pickle['sensitive_parcels']
        print(f"File {i}: {len(sensitive_parcels)} sensitive parcels")
        if len(sensitive_parcels) < min_len_sens_parcels:
            min_len_sens_parcels = len(sensitive_parcels)
            min_len_i = i
            
        # also check how many of the sensitive parcels are motor are
        motor_labels = [l for l in sensitive_parcels if l.startswith("SomMot")]
        print(f"File {i}: {len(motor_labels)} motor labels")
        print("-"*50)
            
min_len_i, min_len_sens_parcels

File 0: 143 sensitive parcels
File 0: 28 motor labels
--------------------------------------------------
File 1: 143 sensitive parcels
File 1: 28 motor labels
--------------------------------------------------
File 2: 143 sensitive parcels
File 2: 28 motor labels
--------------------------------------------------
File 3: 143 sensitive parcels
File 3: 28 motor labels
--------------------------------------------------
File 4: 143 sensitive parcels
File 4: 28 motor labels
--------------------------------------------------
File 5: 143 sensitive parcels
File 5: 28 motor labels
--------------------------------------------------
File 6: 143 sensitive parcels
File 6: 28 motor labels
--------------------------------------------------
File 7: 143 sensitive parcels
File 7: 28 motor labels
--------------------------------------------------
File 8: 143 sensitive parcels
File 8: 28 motor labels
--------------------------------------------------
File 9: 143 sensitive parcels
File 9: 28 motor labels
-

(0, 143)

In [57]:
with open(proc_pkl_files[7], 'rb') as handle:
    data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    sensitive_parcels = data_pickle['sensitive_parcels']
len(sensitive_parcels)

143

In [58]:
from collections import Counter

# Gather all fs_mean values from all files
fs_list = []
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    
    dt = np.diff(delta_brain.time.data)
    fs_mean = float(1.0 / np.mean(dt))
    
    fs_list.append(fs_mean)

fs_counts = Counter(fs_list)
len (proc_pkl_files), fs_counts

(144, Counter({17.438616071428573: 108, 17.438616071428577: 36}))

In [60]:
with open(file, 'rb') as handle:
    data_pickle = pickle.load(handle)

delta_brain = data_pickle['delta_conc']
sensitive_parcels = data_pickle['sensitive_parcels']
rec_stim = data_pickle['rec_stim']

delta_brain.time.data

array([ 20.64384 ,  20.701184,  20.758528, ..., 564.953088, 565.010432,
       565.067776], shape=(9495,))

In [61]:
delta_brain.sel(parcel=template_sens_parcel_list).shape

(9495, 143, 2)

In [62]:
processed_path

PosixPath('datasets/processed/imageRecon_params/vfc_hd/"channel_space_data"/full')

In [ ]:
if not use_channel_space:
    baseline_duration = 2.5  # in seconds
    n_shifts = 9
    duration = 10  # in seconds
    post_padding = 5  # in seconds
    extract_rest_segments = True # for vfc_hd, Anderson_sparse
    resample = False
    delta_range = (-2.5, 2.5)

        
    if DATASET_NAME == "BallSqueezingHD_modified":
        n_timepoints = 87 # 244 # 174 # fixed length after shifting
    elif DATASET_NAME == "BS_Laura":
        n_timepoints = 87 # 244 # 174 # fixed length after shifting
        resample = True
    elif DATASET_NAME == "vfc_hd":
        n_timepoints = 174 # fixed length after shifting


    if DATASET_NAME == "FreshMotor":
        delta_range = (-2.0, 0.0)

    start_shift = np.linspace(*delta_range, n_shifts)
        
    for file in proc_pkl_files:
        with open(file, 'rb') as handle:
            data_pickle = pickle.load(handle)
        
        delta_brain = data_pickle['delta_conc']
        sensitive_parcels = data_pickle['sensitive_parcels']
        rec_stim = data_pickle['rec_stim']

        # [OLD]: Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
        # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
        
        # [NEW]: select only parcels in the common template
        delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)
        
        # ---- NEW: Resampling ---- use for Laura, (and ??)
        if resample:
            if DATASET_NAME == "BS_Laura":
                target_fs = 8.7 # 24.4 # 8.98876404494382  # From BSQ-HD
            dt = 1.0 / target_fs
            t0 = float(delta_brain.time.min())
            t1 = float(delta_brain.time.max())
            new_time = np.arange(t0, t1 + 1e-9, dt)
            delta_brain = delta_brain.interp(time=new_time)
        # -------------------------
        
        # Process event segments
        i = 0
        for index, row in rec_stim.iterrows():
            # Binary labeling: pool word conditions into "task" label
            trial_type_clean = row["trial_type"].lower()

            if "word" in trial_type_clean:
                label = f"{trial_type_clean}_task"
            else:
                label = trial_type_clean
                
            for s in start_shift:
                start_time = row["onset"] + s
                end_time = start_time + duration + post_padding # in seconds
                baseline = delta_brain.sel(
                    time=slice(row["onset"] - baseline_duration, row["onset"])
                ).mean("time")
                
                # Then, trimming is easy with `.sel()`:
                x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
                x = x.isel(time=slice(0, n_timepoints))
                
                # new check
                # print(x.sizes["time"], " timepoints")
                if x.sizes["time"] < n_timepoints:
                    print(f"Skipping segment for file {os.path.basename(file)} at trial {i} due to insufficient length: {x.sizes['time']} < {n_timepoints}")
                    # continue  # skip short segment
                
                x = x.transpose("parcel", "chromo", "time")
                del x.time.attrs['units']

                if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
                    os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
                if s == 0:
                    x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
                    i += 1
                else:
                    x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
                    i += 1

        if extract_rest_segments:
            # --- REST-STATE SEGMENT EXTRACTION ---
            # Rest segments: start at event_onset + 15s, end at next_event_onset - 5s
            # Extracted with 0.5s sliding window, 10s duration each
            recording_start = float(delta_brain.time.values[0])
            recording_end = float(delta_brain.time.values[-1])
            segment_length_sec = duration  # 10 seconds
            rest_label = "rest"
            step_sec = 0.25  # sliding window step
            rest_segment_count = 0
            
            # Process rest intervals between consecutive events
            for idx in range(len(rec_stim)):
                current_onset = rec_stim.iloc[idx]["onset"]
                
                # Rest interval: [event_onset + 15s, next_event_onset - 5s]
                rest_start = current_onset + 15.0
                
                if idx + 1 < len(rec_stim):
                    next_onset = rec_stim.iloc[idx + 1]["onset"]
                    rest_end = next_onset - 5.0
                else:
                    rest_end = recording_end
                
                interval_length = rest_end - rest_start
                
                # Check if interval is long enough for at least one segment
                if interval_length < segment_length_sec:
                    print(f"  Skipping rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s < {segment_length_sec}s)")
                    continue
                
                print(f"  Rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s)")
                
                # First pass: collect valid rest segment starts in this interval
                t = rest_start
                valid_segment_starts = []
                
                while t + segment_length_sec <= rest_end:
                    baseline_start = max(t - baseline_duration, recording_start)
                    baseline_rest = delta_brain.sel(time=slice(baseline_start, t)).mean("time")
                    x_rest = delta_brain.sel(time=slice(t, t + segment_length_sec)) - baseline_rest
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    
                    # Skip if segment is too short
                    if x_rest.sizes["time"] < n_timepoints:
                        print(f"    Skipping rest segment at t={t:.1f}s (insufficient samples: {x_rest.sizes['time']} < {n_timepoints})")
                        t += step_sec
                        continue
                    
                    valid_segment_starts.append(t)
                    t += step_sec
                
                if len(valid_segment_starts) == 0:
                    continue
                
                # Select the segment centered closest to interval midpoint as test
                interval_midpoint = (rest_start + rest_end) / 2.0
                segment_centers = np.array(valid_segment_starts) + (segment_length_sec / 2.0)
                test_segment_idx = int(np.argmin(np.abs(segment_centers - interval_midpoint)))
                
                interval_segments = 0
                for seg_idx, t_start in enumerate(valid_segment_starts):
                    baseline_start = max(t_start - baseline_duration, recording_start)
                    baseline_rest = delta_brain.sel(time=slice(baseline_start, t_start)).mean("time")
                    x_rest = delta_brain.sel(time=slice(t_start, t_start + segment_length_sec)) - baseline_rest
                    x_rest = x_rest.isel(time=slice(0, n_timepoints))
                    x_rest = x_rest.transpose("parcel", "chromo", "time")
                    del x_rest.time.attrs['units']
                    
                    is_test_segment = seg_idx == test_segment_idx
                    rest_suffix = "_test.nc" if is_test_segment else ".nc"
                    rest_filename = file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", f"_{rest_label}_{rest_segment_count}{rest_suffix}")
                    x_rest.to_netcdf(rest_filename)
                    rest_segment_count += 1
                    interval_segments += 1
                
                print(f"  Extracted {interval_segments} rest segments from interval {idx} (middle segment index {test_segment_idx} saved as _test.nc)")
        
            print(f"Total rest segments extracted: {rest_segment_count}")

        print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

    print('\n--- Done!---')

  Rest interval 0: [39.3s - 52.3s] (length: 13.0s)
  Extracted 13 rest segments from interval 0 (middle segment index 6 saved as _test.nc)
  Rest interval 1: [72.3s - 83.4s] (length: 11.0s)
  Extracted 5 rest segments from interval 1 (middle segment index 2 saved as _test.nc)
  Skipping rest interval 2: [103.4s - 111.4s] (length: 8.0s < 10s)
  Rest interval 3: [131.4s - 143.4s] (length: 12.0s)
  Extracted 8 rest segments from interval 3 (middle segment index 4 saved as _test.nc)
  Rest interval 4: [163.4s - 174.4s] (length: 11.0s)
  Extracted 5 rest segments from interval 4 (middle segment index 2 saved as _test.nc)
  Rest interval 5: [194.4s - 205.5s] (length: 11.0s)
  Extracted 5 rest segments from interval 5 (middle segment index 2 saved as _test.nc)
  Rest interval 6: [225.5s - 237.5s] (length: 12.0s)
  Extracted 8 rest segments from interval 6 (middle segment index 4 saved as _test.nc)
  Rest interval 7: [257.5s - 267.5s] (length: 10.0s)
  Extracted 1 rest segments from interval 7

In [63]:
set(["parcel", "chromo", "time"]).issubset(x.dims)

NameError: name 'x' is not defined

In [64]:
# (parcel, chromo, time)

# if DATASET_NAME == "BS_Laura":
file_to_load = "datasets/processed/imageRecon_params/BS_Laura/full/am_0.1__as_0.1/sub-568/sub-568_task-BS_run-01_nirs_left_0.nc"
# elif DATASET_NAME == "vfc_hd":

# time parcel chromo
file_to_load = "datasets/processed/imageRecon_params/vfc_hd/full/am_0.1__as_0.1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_wordcongruent_task_0.nc"

# time parcel chromo
file_to_load = "datasets/processed/imageRecon_params/vfc_hd/full/am_0.1__as_0.1/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_rest_0.nc"

ds = xr.load_dataset(file_to_load)
ds

<xarray.Dataset> Size: 401kB
Dimensions:                        (time: 174, parcel: 143, chromo: 2)
Coordinates:
  * time                           (time) float64 1kB 33.6 33.66 ... 43.47 43.52
    samples                        (time) int32 696B 586 587 588 ... 757 758 759
  * parcel                         (parcel) object 1kB 'ContA_IPS_2_LH' ... '...
  * chromo                         (chromo) object 16B 'HbO' 'HbR'
Data variables:
    __xarray_dataarray_variable__  (parcel, chromo, time) float64 398kB 0.023...

In [ ]:
if DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
    from collections import defaultdict
    segmented_files = glob.glob(str(processed_path / 'am_*__as_*' / 'sub-*' / '*.nc'))

    subject_label_counts = defaultdict(lambda: defaultdict(int))

    for file_path in segmented_files:
        filename = os.path.basename(file_path)
        subject = os.path.basename(os.path.dirname(file_path))
        
        if '_task_' in filename:
            label = 'task'
        elif '_rest_' in filename:
            label = 'rest'
        else:
            label = 'unknown'
        
        subject_label_counts[subject][label] += 1

    data_for_df = []
    for subject in sorted(subject_label_counts.keys()):
        counts = subject_label_counts[subject]
        data_for_df.append({
            'Subject': subject,
            'Task': counts.get('task', 0),
            'Rest': counts.get('rest', 0),
            'Total': counts.get('task', 0) + counts.get('rest', 0)
        })

    df_counts = pd.DataFrame(data_for_df)

    print(df_counts.to_string(index=False))

    print("="*40)
    print(f"Total subjects: {len(df_counts)}")
    print(f"Total task segments: {df_counts['Task'].sum()}")
    print(f"Total rest segments: {df_counts['Rest'].sum()}")
    print(f"Total segments: {df_counts['Total'].sum()}")
    print(f"\nClass balance: Task={df_counts['Task'].sum()}, Rest={df_counts['Rest'].sum()}")
    print(f"Task/Rest ratio: {df_counts['Task'].sum() / df_counts['Rest'].sum():.2f}" if df_counts['Rest'].sum() > 0 else "Task/Rest ratio: N/A")

Subject  Task  Rest  Total
 sub-01   972   756   1728
 sub-06   972   324   1296
 sub-08  1458  1071   2529
 sub-09  1458  1053   2511
 sub-11  1458  1035   2493
 sub-12  1458  1026   2484
 sub-14  1458   963   2421
 sub-15  1458   954   2412
 sub-17  1458   990   2448
 sub-20  1458  1080   2538
 sub-22  1458   990   2448
 sub-23  1458  1053   2511
 sub-24  1458   936   2394
 sub-25  1458   990   2448
 sub-26  1458   972   2430
 sub-27  1458  1044   2502
Total subjects: 16
Total task segments: 22356
Total rest segments: 15237
Total segments: 37593

Class balance: Task=22356, Rest=15237
Task/Rest ratio: 1.47


In [ ]:
if DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
    test_files = sorted(processed_path.rglob("*_test.nc"))

    counts = {"task": 0, "rest": 0}
    for f in test_files:
        name = f.name.lower()
        if "_rest_" in name:
            counts["rest"] += 1
        else:
            counts["task"] += 1

    df_test_counts = pd.DataFrame([
        {"Label": "task", "_test_files": counts["task"]},
        {"Label": "rest", "_test_files": counts["rest"]},
    ])

    display(df_test_counts)
    print(f"Total _test files: {len(test_files)}")


,Label,_test_files
0,task,2484
1,rest,1827


Total _test files: 4311


In [69]:
for file in proc_pkl_files[0:1]:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    
    rec_stim = data_pickle['rec_stim']
    conc_pcr = data_pickle['conc_pcr'] # channel space data

conc_pcr.sizes

Frozen({'chromo': 2, 'channel': 214, 'time': 10606})

<!-- # Channel space segmentation -->

In [ ]:
if use_channel_space:
    # deprecated code for channel space processing. We are now doing image reconstruction and then selecting parcels based on sensitivity, thus we don't need to do this channel space processing anymore. However, I keep this code here for reference and in case we want to compare with channel space results in the future.

    baseline_duration = 2.5  # in seconds
    n_shifts = 9
    duration = 10  # in seconds
    post_padding = 5  # in seconds
    n_timepoints = 87  # fixed length after shifting

    if DATASET_NAME == "BallSqueezingHD_modified":
        delta_range = (-2.5, 2.5)
    elif DATASET_NAME == "FreshMotor":
        delta_range = (-2.0, 0.0)
        
    start_shift = np.linspace(*delta_range, n_shifts)

        
    for file in proc_pkl_files:
        with open(file, 'rb') as handle:
            data_pickle = pickle.load(handle)
        
        rec_stim = data_pickle['rec_stim']
        conc_pcr = data_pickle['conc_pcr'] # channel space data

        # Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
        # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
        

        # ---- NEW: Resampling ---- use for Laura, (and ??)
        if resample:
            if DATASET_NAME == "BS_Laura":
                target_fs = 8.7 # 24.4 # 8.98876404494382  # From BSQ-HD
            dt = 1.0 / target_fs
            t0 = float(conc_pcr.time.min())
            t1 = float(conc_pcr.time.max())
            new_time = np.arange(t0, t1 + 1e-9, dt)
            conc_pcr = conc_pcr.interp(time=new_time)
        # -------------------------
                
        i = 0
        for index, row in rec_stim.iterrows():
            label = row["trial_type"].lower()
            for s in start_shift:
                start_time = row["onset"] + s
                end_time = start_time + duration + post_padding # in seconds
                
                # channel space baseline
                baseline_conc_pcr = conc_pcr.sel(
                    time=slice(row["onset"] - baseline_duration, row["onset"])
                ).mean("time")
            
                # channel space segment
                x_channel = conc_pcr.sel(time=slice(start_time, end_time)) - baseline_conc_pcr
                x_channel = x_channel.isel(time=slice(0, n_timepoints))
                x_channel = x_channel.transpose("channel", "chromo", "time")

                # print(x_channel.shape)
                del x_channel.time.attrs['units']
                if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
                    os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
                if s == 0:
                    x_channel.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
                    i += 1
                else:
                    x_channel.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
                    i += 1

        print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))
    print('--- Done!---')            

In [ ]:
# # load a specific .nc file
# file_to_load = "datasets/processed/Anderson_sparse/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs_wordcongruent_0.nc"
# ds = xr.load_dataset(file_to_load)
# ds

# file_to_load = "datasets/processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_wordcongruent_0.nc"
# ds = xr.load_dataset(file_to_load)
# ds



<!-- ### Old appraoch using event files and freq0.5 -->

<!-- FreshMotor

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (time: 87, channel: 68, chromo: 2)

---
BallSqueezing

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (channel: 100, chromo: 2, time: 87)

---
vfc_hd

- parcel space: (time: 174, parcel: 145, chromo: 2)
- channel space: (channel: 214, wavelength: 2, time: 6712) -->


<!-- # Outdated approach -->

In [ ]:
# baseline_duration = 2.5  # in seconds
# n_shifts = 9
# duration = 10  # in seconds
# post_padding = 5  # in seconds
# n_timepoints = 87  # fixed length after shifting

# if DATASET_NAME == "BallSqueezingHD_modified":
#     delta_range = (-2.5, 2.5)
# elif DATASET_NAME == "FreshMotor":
#     delta_range = (-2.0, 0.0)
# start_shift = np.linspace(*delta_range, n_shifts)






# label_dict = {'right':1, 'left':2}  
# subject_to_rec = {}            
# INDEX = 0     
# freq_dir = processed_path / f'frq{0.5}'
# freq_dir.mkdir(exist_ok=True)       
# for file in proc_pkl_files:
#     with open(file, 'rb') as handle:
#         data_pickle = pickle.load(handle)
    
#     delta_brain = data_pickle['delta_conc']
#     sensitive_parcels = data_pickle['sensitive_parcels']
#     rec_stim = data_pickle['rec_stim']

#     # Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
#     # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
#     delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)

#     i = 0
    
    
    
    
#     # ------
#     SUB = PureWindowsPath(file).parts[-2]
#     if SUB not in subject_to_rec:
#         subject_to_rec[SUB] = []
#     try:
#         sub_dir = Path(freq_dir) / SUB
#         sub_dir.mkdir(parents=True, exist_ok=True)
#     except FileExistsError:
#         pass
#     # ------

#     for index, row in rec_stim.iterrows():
#         label = row["trial_type"].lower()
#         for s in start_shift:
#             start_time = row["onset"] + s
#             end_time = start_time + duration + post_padding # in seconds
#             baseline = delta_brain.sel(
#                 time=slice(row["onset"] - baseline_duration, row["onset"])
#             ).mean("time")
            
#             # Then, trimming is easy with `.sel()`:
#             x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
#             x = x.isel(time=slice(0, n_timepoints))
#             x = x.transpose("parcel", "chromo", "time")
#             del x.time.attrs['units']

#             data = {
#                 'xt': x,
#                 'file': file,
#                 'class': label_dict[label],
#             }

#             # -----
#             # save events
#             filename = r'{}/{}/event_{}_delta{:+.2f}_{}.pkl'.format(freq_dir, SUB, 'aug' if s != 0.0 else 'orig', s, INDEX)
#             subject_to_rec[SUB].append(filename)
#             INDEX += 1
#             # -----
            
            
#             # if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
#             #     os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
#             # if s == 0:
#             #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
#             #     i += 1
#             # else:
#             #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
#             #     i += 1
#             with open(filename, 'wb') as handle:
#                 pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)
                
# with open('{}/meta_event_{}.pkl'.format(freq_dir, 0.5), 'wb') as handle:
#     pickle.dump(subject_to_rec, handle, protocol=pickle.HIGHEST_PROTOCOL)
# print('--- Done!---')            
#     # print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

In [ ]:
# events = str(processed_path / "frq{}" / "meta_event_{}.pkl")
    
# meta_events = []

# with open(events.format(0.5, 0.5), 'rb') as handle:
#     meta = pickle.load(handle)
# meta_events.append(meta)


# if DATASET_NAME == "BallSqueezingHD_modified":
#     session_to_files = {'run-1':[],
#                         'run-2':[],
#                         'run-3':[]}
    
# elif DATASET_NAME == "FreshMotor":    
#     session_to_files = {'run-left2s':[],
#                         'run-right2s':[],
#                         'run-left3s':[],
#                         'run-right3s':[]}

# files_to_session = {}
# for meta_event in meta_events:
#     for sub in meta_event:
#         for file in meta_event[sub]:
#             meta = None
#             with open(file, 'rb') as handle:
#                 meta = pickle.load(handle) 
            
#             # NEW: robust run parsing; safe with parameter-grid folder path layout
#             file_base = os.path.basename(meta['file'])
#             run_match = re.search(r"(run-[^_]+)", file_base)
#             if run_match is None:
#                 raise ValueError(f"Could not parse run token from file name: {file_base}")

#             run = run_match.group(1)
#             files_to_session[file] = run
#             if run not in session_to_files:
#                 session_to_files[run] = []
#             session_to_files[run].append(file)

# for run in session_to_files:
#     print(run, len(session_to_files[run]))

# # this will save the mapping of files to sessions used for LOSO
# with open(processed_path / 'files_to_sessions.pkl', 'wb') as handle:
#     pickle.dump(files_to_session, handle, protocol=pickle.HIGHEST_PROTOCOL)     
# print("Saved files_to_sessions.pkl")

In [ ]:
# # NEW: simple visualization for one recording across recon-parameter views
# # This cell expects you already ran the preprocessing cells and have:
# # rec, c_meas, Adot, get_recon_settings

# import matplotlib.pyplot as plt
# import numpy as np

# required_names = ["rec", "c_meas", "Adot", "get_recon_settings", "dot"]
# missing = [name for name in required_names if name not in globals()]
# if missing:
#     raise RuntimeError(f"Missing required variables for visualization: {missing}")

# # Reconstruct the same recording using all configured augmentation views.
# # This avoids surface-mesh plotting because brain_only=True returns only brain vertices.
# view_imgs = {}
# view_meta = {}
# for view_id, alpha_meas, alpha_spatial, alpha_meas_0, alpha_spatial_0, alpha_meas_multiplier, alpha_spatial_multiplier in get_recon_settings(c_meas):
#     recon_vis = dot.ImageRecon(
#         Adot,
#         recon_mode="mua2conc",
#         brain_only=True,
#         alpha_meas=float(alpha_meas),
#         alpha_spatial=float(alpha_spatial),
#         apply_c_meas=True,
#         spatial_basis_functions=None,
#     )
#     img = recon_vis.reconstruct(rec["od_pcr1"], c_meas)
#     img.time.attrs["units"] = units.s
#     img = img.cd.freq_filter(fmin=0.01, fmax=0.5, butter_order=4)
#     img = img.where(img.is_brain == True)
#     img = img.pint.to("uM").pint.dequantify()

#     view_imgs[view_id] = img
#     view_meta[view_id] = {
#         "alpha_meas": float(alpha_meas),
#         "alpha_spatial": float(alpha_spatial),
#         "alpha_meas_multiplier": float(alpha_meas_multiplier),
#         "alpha_spatial_multiplier": float(alpha_spatial_multiplier),
#     }

# if not view_imgs:
#     raise RuntimeError("No reconstruction views were generated.")

# # Choose one shared representative time from the middle view.
# ref_view = "am_1__as_1" if "am_1__as_1" in view_imgs else next(iter(view_imgs.keys()))
# ref_trace = view_imgs[ref_view].sel(chromo="HbO").mean("vertex", skipna=True)
# t_peak = ref_trace.time.values[np.nanargmax(np.abs(ref_trace.values))]

# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(view_imgs)))

# # Panel 1: mean brain HbO time course per reconstruction setting.
# for color, (view_id, img) in zip(colors, view_imgs.items()):
#     meta = view_meta[view_id]
#     trace = img.sel(chromo="HbO").mean("vertex", skipna=True)
#     axes[0].plot(
#         trace.time.values,
#         trace.values,
#         color=color,
#         lw=2,
#         label=f"{view_id}: am={meta['alpha_meas']:.2e}, as={meta['alpha_spatial']:.1e}",
#     )
# axes[0].axvline(t_peak, color="k", lw=1, ls="--", alpha=0.6)
# axes[0].set_title("Mean brain HbO over time")
# axes[0].set_xlabel("time / s")
# axes[0].set_ylabel("HbO / uM")
# axes[0].grid(True, alpha=0.3)
# axes[0].legend(fontsize=8)

# # Panel 2: brain-vertex HbO distribution at the same peak time.
# labels = []
# distributions = []
# for view_id, img in view_imgs.items():
#     snap = img.sel(chromo="HbO").sel(time=t_peak, method="nearest")
#     vals = snap.values
#     vals = vals[np.isfinite(vals)]
#     labels.append(view_id)
#     distributions.append(vals)

# axes[1].boxplot(distributions, labels=labels, showfliers=False)
# axes[1].axhline(0, color="k", lw=1, alpha=0.5)
# axes[1].set_title(f"Brain-vertex HbO distribution at t={float(t_peak):.2f}s")
# axes[1].set_xlabel("reconstruction view")
# axes[1].set_ylabel("HbO / uM")
# axes[1].grid(True, axis="y", alpha=0.3)

# fig.suptitle("Effect of alpha_meas / alpha_spatial augmentation on one recording")
# fig.tight_layout()
# plt.show()


In [ ]:
# FINAL SUMMARY: dataset and segmentation settings actually used in this notebook
# Run this after preprocessing and segmentation cells.

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd


def _safe_get(name, default=None):
    return globals().get(name, default)


def _subject_from_path(path):
    for part in Path(path).parts:
        if part.startswith("sub-"):
            return part
    return None


def _labels_from_segment_files(paths):
    labels = set()
    known_labels = ["left", "right", "task", "rest", "wordcongruent", "wordincongruent"]
    for path in paths:
        name = Path(path).name.lower()
        for label in known_labels:
            if f"_{label}_" in name:
                labels.add(label)
    return sorted(labels)


def _get_sampling_frequency():
    # Prefer the first raw file so this reports the dataset acquisition frequency.
    raw_files = _safe_get("files", [])
    if raw_files:
        try:
            rec0 = cedalion.io.read_snirf(raw_files[0])[0]
            t = np.asarray(rec0["amp"].time.values, dtype=float)
            if len(t) > 1:
                dt = float(np.median(np.diff(t)))
                return 1.0 / dt, dt
        except Exception as exc:
            print(f"Could not infer sampling frequency from raw file: {exc}")
    return None, None


def _get_first_segment_info(segment_files):
    if not segment_files:
        return None, None, None, None, None, None
    try:
        x = xr.open_dataarray(segment_files[0])
    except Exception:
        x = xr.open_dataset(segment_files[0]).to_array().squeeze()

    spatial_dim = "parcel" if "parcel" in x.dims else "channel" if "channel" in x.dims else None
    spatial_size = int(x.sizes[spatial_dim]) if spatial_dim else None
    chromo_size = int(x.sizes["chromo"]) if "chromo" in x.dims else None
    time_size = int(x.sizes["time"]) if "time" in x.dims else None

    segment_dt_seconds = None
    segment_fs_hz = None
    segment_duration_seconds = None
    if "time" in x.coords and time_size and time_size > 1:
        t = np.asarray(x.time.values, dtype=float)
        segment_dt_seconds = float(np.median(np.diff(t)))
        segment_fs_hz = 1.0 / segment_dt_seconds
        # Duration represented by samples. The coordinate span is one dt shorter.
        segment_duration_seconds = float(time_size * segment_dt_seconds)

    return (
        spatial_dim,
        spatial_size,
        chromo_size,
        time_size,
        segment_fs_hz,
        segment_dt_seconds,
        segment_duration_seconds,
    )


def _processed_raw_files(raw_files):
    # Match the notebook's processing exclusions before summarizing event timing.
    skipped = set(str(p) for p in _safe_get("skipped_subjects", []))
    dataset_name = _safe_get("DATASET_NAME")
    processed = []
    for file in raw_files:
        file = str(file)
        if file in skipped:
            continue
        if dataset_name == "vfc_hd" and "sub-13" in Path(file).parts:
            continue
        processed.append(file)
    return processed


def _get_min_event_onset_diff(raw_files):
    # Compute from the same event tables and standardization used by preprocessing.
    onset_diffs = []
    dataset_name = _safe_get("DATASET_NAME")
    for file in _processed_raw_files(raw_files):
        try:
            rec_i = cedalion.io.read_snirf(file)[0]
            stim_i = cedalion.io.read_events_from_tsv(file.replace("nirs.snirf", "events.tsv"))
            stim_i, rec_i = standardize_trial_types(dataset_name, file, stim_i, rec_i)
            onsets = np.sort(np.asarray(rec_i.stim.onset.values, dtype=float))
            if len(onsets) > 1:
                onset_diffs.extend(np.diff(onsets).tolist())
        except Exception as exc:
            print(f"Could not infer event onset spacing for {file}: {exc}")

    if not onset_diffs:
        return None, 0
    return float(np.min(onset_diffs)), len(onset_diffs)


raw_files = list(_safe_get("files", []))
processed_raw_files = _processed_raw_files(raw_files)
pre_path = Path(_safe_get("pre_processed_path")) if _safe_get("pre_processed_path") is not None else None
proc_path = Path(_safe_get("processed_path")) if _safe_get("processed_path") is not None else None

if proc_path is not None:
    segment_files = sorted(proc_path.glob("am_*__as_*/sub-*/*.nc"))
    if not segment_files:
        segment_files = sorted(proc_path.rglob("*.nc"))
else:
    segment_files = []

if pre_path is not None:
    preprocessed_files = sorted(pre_path.glob("am_*__as_*/sub-*/*.pkl"))
    if not preprocessed_files:
        preprocessed_files = sorted(pre_path.rglob("*.pkl"))
else:
    preprocessed_files = []

raw_subjects = sorted({s for s in (_subject_from_path(p) for p in raw_files) if s})
processed_subjects = sorted({Path(p).parent.name for p in segment_files if Path(p).parent.name.startswith("sub-")})
param_configs = sorted({part for p in segment_files for part in Path(p).parts if part.startswith("am_")})
labels = _labels_from_segment_files(segment_files)

fs_hz, dt_seconds = _get_sampling_frequency()
(
    spatial_dim,
    spatial_size_from_segment,
    chromo_size,
    time_size_from_segment,
    segment_fs_hz,
    segment_dt_seconds,
    segment_duration_seconds,
) = _get_first_segment_info(segment_files)

channel_size = len(_safe_get("subset_channels", [])) if _safe_get("subset_channels", None) is not None else None
parcel_template = _safe_get("template_sens_parcel_list", None)
parcel_size = len(parcel_template) if parcel_template is not None else None

n_times = _safe_get("n_timepoints", time_size_from_segment)
final_duration_seconds = segment_duration_seconds
if final_duration_seconds is None and n_times is not None:
    effective_fs = _safe_get("target_fs") if _safe_get("resample") else fs_hz
    final_duration_seconds = (float(n_times) / effective_fs) if effective_fs else None
min_event_onset_diff, n_event_onset_diffs = _get_min_event_onset_diff(raw_files)

summary = {
    "dataset_name": _safe_get("DATASET_NAME"),
    "augmentation_strategy": _safe_get("augmentation_strategy"),
    "subset_type": _safe_get("subset_type"),
    "raw_path": str(_safe_get("raw_path")),
    "pre_processed_path": str(pre_path),
    "processed_path": str(proc_path),
    "n_subjects_raw": len(raw_subjects),
    "n_subjects_processed": len(processed_subjects),
    "n_total_recordings_raw": len(raw_files),
    "n_total_recordings_processed_input": len(processed_raw_files),
    "n_preprocessed_recording_views": len(preprocessed_files),
    "n_segment_files_total": len(segment_files),
    "n_param_configs": len(param_configs),
    "param_configs": ", ".join(param_configs) if param_configs else "n/a",
    "channel_size": channel_size,
    "spatial_dim_in_segment": spatial_dim,
    "spatial_size_in_segment": spatial_size_from_segment,
    "parcel_template_size": parcel_size,
    "chromo_size": chromo_size,
    "raw_sampling_frequency_hz": f"{fs_hz:.3f}" if fs_hz else "n/a",
    "raw_sampling_dt_seconds": f"{dt_seconds:.6f}" if dt_seconds else "n/a",
    "resample": _safe_get("resample"),
    "target_fs_hz": _safe_get("target_fs"),
    "segment_sampling_frequency_hz": f"{segment_fs_hz:.3f}" if segment_fs_hz else "n/a",
    "segment_sampling_dt_seconds": f"{segment_dt_seconds:.6f}" if segment_dt_seconds else "n/a",
    "extract_rest_segments": _safe_get("extract_rest_segments"),
    "min_event_onset_diff_seconds": f"{min_event_onset_diff:.3f}" if min_event_onset_diff is not None else "n/a",
    "n_event_onset_diffs_used": n_event_onset_diffs,
    "n_times_per_segment_setting": n_times,
    "n_times_per_segment_from_file": time_size_from_segment,
    "final_segment_duration_seconds": f"{final_duration_seconds:.2f}" if final_duration_seconds else "n/a",
    "duration_setting_seconds": _safe_get("duration"),
    "post_padding_seconds": _safe_get("post_padding"),
    "baseline_duration_seconds": _safe_get("baseline_duration"),
    "delta_range": str(_safe_get("delta_range")),
    "number_of_shifts": _safe_get("n_shifts"),
    "start_shifts": np.array2string(_safe_get("start_shift"), precision=3) if _safe_get("start_shift", None) is not None else "n/a",
    "label_names": ", ".join(labels) if labels else "n/a",
}

summary_df = pd.DataFrame([summary]).T.reset_index()
summary_df.columns = ["field", "value"]

print("Final dataset / segmentation summary")
print("=" * 44)
display(summary_df)
# save data frame
summary_df.to_csv(proc_path / "dataset_segmentation_summary.csv", index=False)


Final dataset / segmentation summary


,field,value
0,dataset_name,vfc_hd
1,augmentation_strategy,imageRecon_params
2,subset_type,full
3,raw_path,datasets/raw/vfc_hd
4,pre_processed_path,datasets/pre_processed/imageRecon_params/vfc_h...
5,processed_path,datasets/processed/imageRecon_params/vfc_hd/full
6,n_subjects_raw,17
7,n_subjects_processed,16
8,n_total_recordings_raw,17
9,n_total_recordings_processed_input,16
